# Task 3: Baseline model

In [1]:
!nvidia-smi

Wed Mar  4 10:57:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   42C    P8              3W /   65W |    4192MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers torch scikit-learn pandas requests


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from urllib import request
import requests
import pandas as pd
import logging
import torch
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report
import numpy as np

c:\Users\sfais\projects\nlp_cw\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# prepare logger
logging.basicConfig(level=logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

# check gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


# Download Data

In [5]:
for url, fname in [
    ("https://raw.githubusercontent.com/CRLala/NLPLabs-2024/main/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv", "dontpatronizeme_pcl.tsv"),
    ("https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/dev_semeval_parids-labels.csv", "dev_semeval_parids-labels.csv"),
    ("https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/train_semeval_parids-labels.csv", "train_semeval_parids-labels.csv"),
]:
    r = requests.get(url)
    open(fname, "wb").write(r.content)
    print(f"Downloaded {fname}")

module_url = "https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/dont_patronize_me.py"
with request.urlopen(module_url) as f, open("dont_patronize_me.py", "w") as out:
    out.write(f.read().decode("utf-8"))
print("Downloaded dont_patronize_me.py")

Downloaded dontpatronizeme_pcl.tsv
Downloaded dev_semeval_parids-labels.csv
Downloaded train_semeval_parids-labels.csv
Downloaded dont_patronize_me.py


# Fetch "Don't Patronize Me!" data manager module

In [6]:
from dont_patronize_me import DontPatronizeMe

In [7]:
dpm = DontPatronizeMe('.', '.')

In [8]:
dpm.load_task1()

In [9]:
# helper function to save predictions to an output file
def labels2file(p, outf_path):
    with open(outf_path,'w') as outf:
        for pi in p:
            outf.write(','.join([str(k) for k in pi])+'\n')

# Load paragraph IDs

In [10]:
trids = pd.read_csv('train_semeval_parids-labels.csv')
teids = pd.read_csv('dev_semeval_parids-labels.csv')
trids.par_id = trids.par_id.astype(str)
teids.par_id = teids.par_id.astype(str)

# Rebuild training set

In [11]:
data = dpm.train_task1_df

rows = []  # will contain par_id, label and text
for idx in range(len(trids)):
    parid = trids.par_id[idx]
    keyword = data.loc[data.par_id == parid].keyword.values[0]
    text = data.loc[data.par_id == parid].text.values[0]
    label = data.loc[data.par_id == parid].label.values[0]
    rows.append({
        'par_id': parid,
        'community': keyword,
        'text': text,
        'label': label
    })

trdf1 = pd.DataFrame(rows)
trdf1

,par_id,community,text,label
0,4341,poor-families,"The scheme saw an estimated 150,000 children f...",1
1,4136,homeless,Durban 's homeless communities reconciliation ...,1
2,10352,poor-families,The next immediate problem that cropped up was...,1
3,8279,vulnerable,Far more important than the implications for t...,1
4,1164,poor-families,To strengthen child-sensitive social protectio...,1
...,...,...,...,...
8370,8380,refugee,Rescue teams search for survivors on the rubbl...,0
8371,8381,hopeless,The launch of ' Happy Birthday ' took place la...,0
8372,8382,homeless,"The unrest has left at least 20,000 people dea...",0
8373,8383,hopeless,You have to see it from my perspective . I may...,0


# Rebuild test set

In [12]:
rows = []  # will contain par_id, label and text
for idx in range(len(teids)):
    parid = teids.par_id[idx]
    keyword = data.loc[data.par_id == parid].keyword.values[0]
    text = data.loc[data.par_id == parid].text.values[0]
    label = data.loc[data.par_id == parid].label.values[0]
    rows.append({
        'par_id': parid,
        'community': keyword,
        'text': text,
        'label': label
    })

tedf1 = pd.DataFrame(rows)
print(f"Test set size: {len(tedf1)}")
tedf1

Test set size: 2094


,par_id,community,text,label
0,4046,hopeless,We also know that they can benefit by receivin...,1
1,1279,refugee,Pope Francis washed and kissed the feet of Mus...,1
2,8330,refugee,Many refugees do n't want to be resettled anyw...,1
3,4063,in-need,"""Budding chefs , like """" Fred """" , """" Winston ...",1
4,4089,homeless,"""In a 90-degree view of his constituency , one...",1
...,...,...,...,...
2089,10462,homeless,"The sad spectacle , which occurred on Saturday...",0
2090,10463,refugee,""""""" The Pakistani police came to our house and...",0
2091,10464,disabled,"""When Marie O'Donoghue went looking for a spec...",0
2092,10465,women,"""Sri Lankan norms and culture inhibit women fr...",0


# Fine-tune facebook/roberta-base

In [13]:
# Downsample negative instances (2:1 neg:pos ratio)
pcldf = trdf1[trdf1.label == 1]
npos = len(pcldf)
training_set1 = pd.concat([pcldf, trdf1[trdf1.label == 0][:npos * 2]]).reset_index(drop=True)
print(f"Training set size: {len(training_set1)} | PCL: {npos} | Non-PCL: {npos*2}")

Training set size: 2382 | PCL: 794 | Non-PCL: 1588


In [14]:
class PCLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [15]:
MODEL_NAME = 'FacebookAI/roberta-base'
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(device)
print('Model loaded:', MODEL_NAME)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/FacebookAI/roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/FacebookAI/roberta-base/e2da8e2f811d1448a5b465c236feacd80ffbac7b/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/FacebookAI/roberta-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/FacebookAI/roberta-base/e2da8e2f811d1448a5b465c236feacd80ffbac7b/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://hugg

Model loaded: FacebookAI/roberta-base


In [16]:
BATCH_SIZE = 16
MAX_LEN = 128
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5

train_dataset = PCLDataset(
    texts=training_set1.text.tolist(),
    labels=training_set1.label.tolist(),
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = PCLDataset(
    texts=tedf1.text.tolist(),
    labels=tedf1.label.tolist(),
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")

Train batches: 149 | Test batches: 131


In [17]:
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

In [18]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
    return total_loss / len(loader)


def eval_model(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return np.array(all_preds), np.array(all_labels)


print('Starting training...')
for epoch in range(NUM_EPOCHS):
    avg_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    preds, true_labels = eval_model(model, test_loader, device)
    f1 = f1_score(true_labels, preds, average='binary')
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | Dev F1: {f1:.4f}")

Starting training...
Epoch 1/3 | Loss: 0.5706 | Dev F1: 0.4110
Epoch 2/3 | Loss: 0.3952 | Dev F1: 0.5008
Epoch 3/3 | Loss: 0.2782 | Dev F1: 0.5122


## Predictions

In [19]:
preds_task1, true_labels = eval_model(model, test_loader, device)
print(classification_report(true_labels, preds_task1, target_names=['Non-PCL', 'PCL']))
print('Prediction distribution:', Counter(preds_task1))

              precision    recall  f1-score   support

     Non-PCL       0.97      0.88      0.92      1895
         PCL       0.39      0.74      0.51       199

    accuracy                           0.87      2094
   macro avg       0.68      0.81      0.72      2094
weighted avg       0.91      0.87      0.88      2094

Prediction distribution: Counter({np.int64(0): 1719, np.int64(1): 375})


In [20]:
labels2file([[k] for k in preds_task1], 'base.txt')
print('Predictions saved to base.txt')
!head -n 10 base.txt

Predictions saved to base.txt


'head' is not recognized as an internal or external command,
operable program or batch file.
